# FireSight E0 control (25% list, no added negatives)

Fresh `yolo11s.pt`, 20 epochs, patience 10, checkpoint selection by recall at FPR<=2% on val_cal.
This is the control run only. E1 is a separate kernel after E0 is reviewed.
No TEST/SEALED, no Sample_videos.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "GPU required"
print("gpus", torch.cuda.device_count())

In [ ]:
import importlib.metadata as md, os
os.environ["YOLO_AUTOINSTALL"]="False"; os.environ["YOLO_VERBOSE"]="False"; os.environ["YOLO_OFFLINE"]="True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
def ver(n):
    try: return md.version(n)
    except Exception: return None
if ver("ultralytics") != "8.4.147":
    get_ipython().system("pip install -q --no-deps --progress-bar off ultralytics==8.4.147 ultralytics-thop")
print({"ultralytics": ver("ultralytics"), "torch": ver("torch")})

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("manifest_summary.json"))[0].parent
CODE = sorted(Path("/kaggle/input").rglob("stage_b_train.py"))[0].parent
YOLO = sorted(Path(CODE).glob("yolo11s.pt"))[0]
print("DATA", DATA); print("CODE", CODE); print("YOLO", YOLO)

In [ ]:
from pathlib import Path
def resolve(line, root):
    p = Path(line.strip())
    return p if p.is_absolute() else (Path(root)/p).resolve()
out = Path("/kaggle/working/lists"); out.mkdir(parents=True, exist_ok=True)
base = [resolve(l, DATA) for l in open(DATA/"lists/train_25.txt", encoding="utf-8") if l.strip()]
val  = [resolve(l, DATA) for l in open(DATA/"lists/val_early.txt", encoding="utf-8") if l.strip()]
cal  = [resolve(l, DATA) for l in open(DATA/"lists/val_cal.txt", encoding="utf-8") if l.strip()]
rep  = [resolve(l, DATA) for l in open(DATA/"lists/val_report.txt", encoding="utf-8") if l.strip()]
for name, items in [("e0_train.txt",base),("val.txt",val),("val_cal.txt",cal),("val_report.txt",rep)]:
    (out/name).write_text("\n".join(str(p) for p in items)+"\n", encoding="utf-8")
    print(name, len(items))
assert len(base)==14747

In [ ]:
import subprocess
cmd = ["python","-u",str(Path(CODE)/"stage_b_train.py"),
       "--dataset",str(DATA),"--train-list",str(Path("/kaggle/working/lists/e0_train.txt")),
       "--val-list",str(Path("/kaggle/working/lists/val.txt")),
       "--val-cal-list",str(Path("/kaggle/working/lists/val_cal.txt")),
       "--out","/kaggle/working/runs/E0","--model",str(YOLO),
       "--epochs","20","--patience","10","--imgsz","640","--batch","64",
       "--device","0","--workers","4","--lr0","0.01","--lrf","0.01","--seed","0","--amp"]
print("RUN", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import subprocess, json
preds="/kaggle/working/preds_E0.jsonl"
subprocess.run(["python",str(Path(CODE)/"stage_b_predict.py"),"--model","/kaggle/working/runs/E0/weights/best_recall.pt",
                "--list","/kaggle/working/lists/val_report.txt","--out",preds,"--device","0","--half"],check=True)
for t in ("0.009","0.015","0.02","0.03"):
    subprocess.run(["python",str(Path(CODE)/"pilot_eval.py"),"--dataset",str(DATA),"--preds",preds,
                    "--model","E0","--target-fpr",t,"--out",f"/kaggle/working/metrics_E0_{t}.json"],check=True)
print(open("/kaggle/working/runs/E0/run_summary.json").read())

In [ ]:
for t in ("0.009","0.015","0.02","0.03"):
    m=json.load(open(f"/kaggle/working/metrics_E0_{t}.json"))
    c=m["splits"]["val"]["combined"]; f=m["splits"]["val"]["fpr_per_negative_image"]
    print(t, "thr",m["thresholds"],"recall",round(c["recall"],4),"fpr",round(f["fpr"],4),
          "tp",c.get("tp"),"fp",f.get("fp"),"n_neg",f.get("n_negatives"))
print("P25 reference: recall 0.9253 @ val_report FPR 3.25%")